# Big Data Analytics

Group 13

- Beatris Daicu, 20221854
- Diogo Carvalho, 2022
- Ricardo Pereira, 2025
- Yehor Malakhov, 2022


## Imports


In [1]:
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, unix_timestamp, when

In [2]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk/"
spark = SparkSession.builder.appName("cab-data").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/02 22:17:21 WARN Utils: Your hostname, cachyos-desktop, resolves to a loopback address: 127.0.1.1; using 192.168.1.80 instead (on interface enp14s0)
26/06/02 22:17:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/02 22:17:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Importing our dataset

For the DataFrames and SQL section of the project, we chose to work with an NYC
Taxi dataset. This dataset was selected because it provides a rich and diverse
set of attributes, making it well-suited for performing a wide variety of
analytical queries and data processing operations. Additionally, its large scale
and real-world nature make it appropriate for demonstrating DataFrame
manipulations, Spark SQL queries, aggregations, joins, and data cleaning
techniques.


DataFrames -> Cleaning \
SQL -> Transformations and queries


Reading CSV


In [3]:
df = spark.read.csv(
    "../data/revenue-for-cab-drivers.csv", header=True, inferSchema=True
)

df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       1| 2020-01-01 00:28:15|  2020-01-01 00:33:03|              1|          1.2|         1|                 N|         238|         239|           1|        6.0|  3.0|    0.5|      1.47|         0.0|                  0.3

Making sure everything is in the correct type


In [4]:
df.printSchema()
df.show(10)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+---

Counting how many distinct values exist "mta_tax", "tolls_amount" and
"improvement_surcharge"


In [5]:
mta_tax_distinct_count = df.select("mta_tax").distinct().count()
print(f"Distinct count for mta_tax: {mta_tax_distinct_count}")
if mta_tax_distinct_count == 1:
    print(f"  -> mta_tax is always the same: {df.select('mta_tax').first()[0]}")
else:
    print("  -> mta_tax is not always the same.")


tolls_amount_distinct_count = df.select("tolls_amount").distinct().count()
print(f"Distinct count for tolls_amount: {tolls_amount_distinct_count}")
if tolls_amount_distinct_count == 1:
    print(
        f"  -> tolls_amount is always the same: {df.select('tolls_amount').first()[0]}"
    )
else:
    print("  -> tolls_amount is not always the same.")


improvement_surcharge_distinct_count = (
    df.select("improvement_surcharge").distinct().count()
)
print(
    f"Distinct count for improvement_surcharge: {improvement_surcharge_distinct_count}"
)
if improvement_surcharge_distinct_count == 1:
    print(
        f"  -> improvement_surcharge is always the same: {df.select('improvement_surcharge').first()[0]}"
    )
else:
    print("  -> improvement_surcharge is not always the same.")

Distinct count for mta_tax: 11
  -> mta_tax is not always the same.
Distinct count for tolls_amount: 1035
  -> tolls_amount is not always the same.
Distinct count for improvement_surcharge: 3
  -> improvement_surcharge is not always the same.


Removing impossible trips, trips that the dropoff time is earlier than the
pickup time. Distance must not be negative and the passenger count has to be
from 1 to 6.


In [6]:
df = df.filter(
    (col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))
    & (col("trip_distance") >= 0)
    & (col("passenger_count").between(1, 6))
)

Fixing negative surcharges, transforming them in NULL


In [7]:
df = df.withColumn(
    "congestion_surcharge",
    when(col("congestion_surcharge") < 0, None).otherwise(
        col("congestion_surcharge")
    ),
)

Removing Extreme Outliers with thresholds


In [8]:
cols = ["trip_distance", "fare_amount"]
bounds = {}
for c in cols:
    q1, q3 = df.approxQuantile(c, [0.25, 0.75], 0.01)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    bounds[c] = (lower, upper)

In [ ]:
df = df.filter(
    (
        col("trip_distance").between(
            bounds["trip_distance"][0], bounds["trip_distance"][1]
        )
    )
    & (
        col("fare_amount").between(
            bounds["fare_amount"][0], bounds["fare_amount"][1]
        )
    )
)

Saving it as a table for sql


In [10]:
df.write.mode("overwrite").saveAsTable("taxi_silver", mode="overwrite")

26/06/02 22:17:31 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/02 22:17:31 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/06/02 22:17:31 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/06/02 22:17:31 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/06/02 22:17:31 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/06/02 22:17:31 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/06/02 22:17:31 WARN MemoryManager: Total allocation exceeds 95.

Checking if the table was correctly created


In [11]:
spark.sql("SHOW TABLES LIKE 'taxi_silver'").show()

+---------+-----------+-----------+
|namespace|  tableName|isTemporary|
+---------+-----------+-----------+
|  default|taxi_silver|      false|
+---------+-----------+-----------+



In [12]:
spark.sql("SELECT * FROM taxi_silver LIMIT 10").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       2| 2020-01-03 13:41:10|  2020-01-03 13:53:01|              1|         1.64|         1|                 N|         186|         162|           1|        9.0|  0.0|    0.5|      2.46|         0.0|                  0.3

In [13]:
spark.sql("SELECT COUNT(*) FROM taxi_silver").show()

+--------+
|count(1)|
+--------+
| 5454616|
+--------+



## SQL Transformations


Creating Analytical View to make analysis easier and to avoid repeatedly
calculating the same derived fields


In [14]:
spark.sql("""CREATE OR REPLACE VIEW taxi_analytics AS
SELECT
  *,
  (unix_timestamp(tpep_dropoff_datetime) -
   unix_timestamp(tpep_pickup_datetime)) / 60 AS trip_duration_minutes,
  HOUR(tpep_pickup_datetime) AS pickup_hour,
  DAYOFWEEK(tpep_pickup_datetime) AS pickup_dow
FROM taxi_silver;""")

DataFrame[]

In [15]:
spark.sql("SELECT * FROM taxi_analytics LIMIT 10").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+---------------------+-----------+----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|trip_duration_minutes|pickup_hour|pickup_dow|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+---------------------+-----------+----------+
|       2| 2020-01-03 13:41:10|  2020-01-03 13:53:01|              1|         1.64|     

When are taxis most used and how much it costs?


In [16]:
spark.sql("""
SELECT
    pickup_hour,
    COUNT(*) AS total_trips,
    ROUND(SUM(total_amount), 2) AS total_revenue,
    ROUND(AVG(total_amount), 2) AS avg_trip_value
FROM taxi_analytics
GROUP BY pickup_hour
ORDER BY avg_trip_value DESC
""").show()

+-----------+-----------+-------------+--------------+
|pickup_hour|total_trips|total_revenue|avg_trip_value|
+-----------+-----------+-------------+--------------+
|         17|     347660|   5228366.94|         15.04|
|         18|     384600|   5785775.75|         15.04|
|         22|     261164|   3916327.22|          15.0|
|         21|     298279|    4429497.9|         14.85|
|         19|     345265|   5123419.95|         14.84|
|         23|     190521|   2815636.55|         14.78|
|         16|     302313|   4436370.18|         14.67|
|          1|     102242|   1498646.45|         14.66|
|         20|     298329|    4371970.4|         14.65|
|          0|     137936|   2020109.08|         14.65|
|          2|      74909|   1088350.73|         14.53|
|          3|      50497|    723036.01|         14.32|
|          9|     265789|   3807040.83|         14.32|
|          8|     263219|   3726409.65|         14.16|
|         10|     261079|   3673640.73|         14.07|
|         

According to the table above, we can understand why a trip at 5 am is most
expensive, probably due to the fact that there are less taxis in service.


Which payment method tips more? \
1 - Credit Card \
2- Cash \
3 - No Charge \
4 - Dispute \
5 - Unknown


In [17]:
spark.sql("""
SELECT
    payment_type,
    COUNT(*) AS trips,
    ROUND(AVG(tip_amount), 2) AS avg_tip,
    ROUND(AVG(total_amount), 2) AS avg_total
FROM taxi_analytics
GROUP BY payment_type
ORDER BY payment_type Asc
""").show()

+------------+-------+-------+---------+
|payment_type|  trips|avg_tip|avg_total|
+------------+-------+-------+---------+
|           1|4048880|   2.34|    15.19|
|           2|1372388|    0.0|     12.2|
|           3|  23525|    0.0|     7.74|
|           4|   9822|    0.0|     6.08|
|           5|      1|    0.0|      0.0|
+------------+-------+-------+---------+



Even though 3 is "No Charge", doesn't mean that there wasn't a payment, just
means the trip wasn't billed normally.


There is 1 Unknown, I don't know if I should delete


Most Popular Pickup Locations


In [18]:
spark.sql("""
SELECT
    PULocationID,
    COUNT(*) AS trip_count
FROM taxi_analytics
GROUP BY PULocationID
ORDER BY trip_count DESC
LIMIT 20
""").show()

+------------+----------+
|PULocationID|trip_count|
+------------+----------+
|         237|    275545|
|         236|    255367|
|         161|    254495|
|         162|    212486|
|         186|    209280|
|         230|    199661|
|         142|    180248|
|         170|    176210|
|         234|    174764|
|          48|    173846|
|         239|    162192|
|         163|    159110|
|         141|    147143|
|          79|    144551|
|          68|    136082|
|         107|    131948|
|         164|    127167|
|         238|    124060|
|         229|    115555|
|         263|    114049|
+------------+----------+



Zone 237 is the most popular. However, as we don't know what zone 237 is, we
will use another csv that maps the numbers of the zones to their names


In [19]:
zones = spark.read.csv(
    "../data/taxi_zone_lookup.csv", header=True, inferSchema=True
)

In [20]:
zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [21]:
zones.write.mode("overwrite").saveAsTable("taxi_zones")

In [22]:
spark.sql("""
SELECT *
FROM taxi_zones
WHERE LocationID = 237
""").show()

+----------+---------+--------------------+------------+
|LocationID|  Borough|                Zone|service_zone|
+----------+---------+--------------------+------------+
|       237|Manhattan|Upper East Side S...| Yellow Zone|
+----------+---------+--------------------+------------+



Upper East Side South, Manhattan. Upper East Side is a wealthy residential
district, meaning people who live here usually catch taxis due to being
financially comfortable.


In [23]:
spark.sql("""
CREATE OR REPLACE VIEW taxi_enriched AS
SELECT
    t.*,
    z.Zone AS pickup_zone,
    z.Borough AS pickup_borough
FROM taxi_analytics t
LEFT JOIN taxi_zones z
ON t.PULocationID = z.LocationID;""")

DataFrame[]

Now, after joining the respective dataset, we can know the names of the pick up
and drop off zones.


In [24]:
spark.sql("SELECT * FROM taxi_enriched LIMIT 10").show(truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+---------------------+-----------+----------+-----------------------------+--------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|trip_duration_minutes|pickup_hour|pickup_dow|pickup_zone                  |pickup_borough|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+---------------------+-----------+----------

### Which taxi system processed more trips and generated more revenue?


In [25]:
spark.sql("""
SELECT
    VendorID,
    COUNT(*) AS trips,
    ROUND(SUM(total_amount), 2) AS total_revenue,
    ROUND(AVG(tip_amount), 2) AS avg_tip,
    ROUND(
        COUNT(*) / COUNT(DISTINCT TO_DATE(tpep_pickup_datetime)),
        0
    ) AS avg_trips_per_day
FROM taxi_enriched
GROUP BY VendorID
ORDER BY total_revenue DESC
""").show()

+--------+-------+-------------+-------+-----------------+
|VendorID|  trips|total_revenue|avg_tip|avg_trips_per_day|
+--------+-------+-------------+-------+-----------------+
|       2|3700382|5.322643537E7|   1.75|          74008.0|
|       1|1754234| 2.52490945E7|   1.71|          56588.0|
+--------+-------+-------------+-------+-----------------+



### Which weekdays are busiest?


In [26]:
spark.sql("""
SELECT
    pickup_dow,
    COUNT(*) AS trips,
    ROUND(AVG(total_amount), 2) AS avg_fare
FROM taxi_enriched
GROUP BY pickup_dow
ORDER BY trips DESC
""").show()

+----------+------+--------+
|pickup_dow| trips|avg_fare|
+----------+------+--------+
|         6|960493|   14.58|
|         5|944001|   14.68|
|         4|895894|   14.57|
|         3|726178|    14.5|
|         7|709878|   14.09|
|         2|619525|   14.11|
|         1|598647|   13.83|
+----------+------+--------+



### Traffic Congestion Impact


Traffic Congestion analysis is important because it connects movement data to
real economic, operational and urban planning decisions.


In [27]:
spark.sql("""
SELECT
    pickup_hour,
    ROUND(AVG(trip_duration_minutes), 2) AS avg_duration,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(trip_distance / trip_duration_minutes), 3) AS miles_per_minute,
    ROUND(AVG(trip_duration_minutes / trip_distance), 3) AS minutes_per_mile
FROM taxi_enriched
WHERE trip_distance > 0 AND trip_duration_minutes > 0
GROUP BY pickup_hour
ORDER BY miles_per_minute DESC
""").show()

+-----------+------------+------------+----------------+----------------+
|pickup_hour|avg_duration|avg_distance|miles_per_minute|minutes_per_mile|
+-----------+------------+------------+----------------+----------------+
|          5|       10.95|        1.89|           0.276|           6.737|
|          4|       12.47|        2.17|            0.27|           7.597|
|          3|        12.7|        2.13|           0.252|           7.909|
|          6|       10.35|         1.8|           0.239|            6.76|
|          2|       13.17|        2.11|           0.237|           7.884|
|          1|       13.61|        2.09|           0.228|           8.173|
|          0|        13.3|        2.06|           0.222|           8.376|
|         23|        13.1|        2.04|           0.215|           7.884|
|         22|       13.05|        2.02|           0.202|            7.88|
|         21|       12.78|        1.95|           0.194|           7.977|
|          7|       12.64|        1.75

More minutes means more congestion. In this case, we can understand that 4 pm
has the biggest average duration, probably because it's the beginning of rush
hour.


### What boroughs generate the most money?


We want to check which parts of NYC generate the best revenue.


In [28]:
spark.sql("""
SELECT
    CASE
        WHEN pickup_borough IN ('Unknown', 'N/A') THEN 'Other'
        ELSE pickup_borough
    END AS borough_group,
    COUNT(*) AS trips,
    ROUND(SUM(total_amount), 2) AS revenue,
    ROUND(AVG(total_amount), 2) AS avg_trip_price
FROM taxi_enriched
GROUP BY
    CASE
        WHEN pickup_borough IN ('Unknown', 'N/A') THEN 'Other'
        ELSE pickup_borough
    END
ORDER BY revenue DESC
""").show()

+-------------+-------+-------------+--------------+
|borough_group|  trips|      revenue|avg_trip_price|
+-------------+-------+-------------+--------------+
|    Manhattan|5293243|7.638114143E7|         14.43|
|       Queens|  81982|   1019345.65|         12.43|
|     Brooklyn|  39768|    546813.99|         13.75|
|        Other|  35052|    469816.76|          13.4|
|        Bronx|   4429|      55211.0|         12.47|
|          EWR|     81|      2427.88|         29.97|
|Staten Island|     61|       773.16|         12.67|
+-------------+-------+-------------+--------------+



EWR = Newark Liberty International Airport


Manhattan has one of the highest concentrations of office workers, hotel,
restaurants, entertainment venues, etc. So this result is not suprising.


## Aggregating Important Values


We are going to aggregate all important values for the pickup boroughs and
dropoff boroughs


In [29]:
spark.sql("""
SELECT
    pickup_borough,
    COUNT(*) AS total_trips,
    ROUND(SUM(total_amount), 2) AS total_revenue,
    ROUND(AVG(total_amount), 2) AS avg_fare,
    ROUND(AVG(trip_duration_minutes), 2) AS avg_duration,
    ROUND(MAX(total_amount), 2) AS max_trip_value,
    ROUND(MIN(total_amount), 2) AS min_trip_value
FROM taxi_enriched
GROUP BY pickup_borough
ORDER BY total_revenue DESC
""").show()

+--------------+-----------+-------------+--------+------------+--------------+--------------+
|pickup_borough|total_trips|total_revenue|avg_fare|avg_duration|max_trip_value|min_trip_value|
+--------------+-----------+-------------+--------+------------+--------------+--------------+
|     Manhattan|    5293243|7.638114143E7|   14.43|       13.34|        1110.8|         -15.8|
|        Queens|      81982|   1019345.65|   12.43|        12.2|         153.3|        -10.92|
|      Brooklyn|      39768|    546813.99|   13.75|       14.58|         101.8|          -6.8|
|       Unknown|      34333|    459517.96|   13.38|       12.76|          78.3|          -8.8|
|         Bronx|       4429|      55211.0|   12.47|       13.98|         123.8|          -6.3|
|           N/A|        719|      10298.8|   14.32|        7.39|         261.8|        -18.05|
|           EWR|         81|      2427.88|   29.97|        0.97|         192.0|          -4.3|
| Staten Island|         61|       773.16|   12.67

Since we didn't have the dropoff zone and borough properly mapped, we will run
the following:


In [30]:
spark.sql("""
CREATE OR REPLACE VIEW taxi_enriched AS
SELECT
    t.*,
    z1.Zone AS pickup_zone,
    z1.Borough AS pickup_borough,
    z2.Zone AS dropoff_zone,
    z2.Borough AS dropoff_borough
FROM taxi_analytics t
LEFT JOIN taxi_zones z1
    ON t.PULocationID = z1.LocationID
LEFT JOIN taxi_zones z2
    ON t.DOLocationID = z2.LocationID
""")

DataFrame[]

Checking if it was properly mapped


In [31]:
spark.sql("""
Select * from taxi_enriched
""").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+---------------------+-----------+----------+--------------------+--------------+--------------------+---------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|trip_duration_minutes|pickup_hour|pickup_dow|         pickup_zone|pickup_borough|        dropoff_zone|dropoff_borough|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+---------

In [32]:
spark.sql("""SELECT
    dropoff_borough,
    COUNT(*) AS total_trips,
    ROUND(SUM(total_amount), 2) AS total_revenue,
    ROUND(AVG(total_amount), 2) AS avg_fare,
    ROUND(AVG(trip_duration_minutes), 2) AS avg_duration,
    ROUND(MAX(total_amount), 2) AS max_trip_value,
    ROUND(MIN(total_amount), 2) AS min_trip_value
FROM taxi_enriched
GROUP BY dropoff_borough
ORDER BY total_revenue DESC""").show()

+---------------+-----------+-------------+--------+------------+--------------+--------------+
|dropoff_borough|total_trips|total_revenue|avg_fare|avg_duration|max_trip_value|min_trip_value|
+---------------+-----------+-------------+--------+------------+--------------+--------------+
|      Manhattan|    5201599|7.419199605E7|   14.26|       13.16|        1110.8|         -15.8|
|       Brooklyn|     102880|   2030479.31|   19.74|       19.97|        113.92|          -8.3|
|         Queens|     113575|    1759290.8|   15.49|       14.73|         153.3|        -10.92|
|        Unknown|      25236|     315700.6|   12.51|       13.76|         104.1|         -9.36|
|          Bronx|      10138|    154824.33|   15.27|        16.2|         499.8|          -6.3|
|            N/A|       1048|     19792.89|   18.89|       12.56|         261.8|        -18.05|
|            EWR|         75|      2529.41|   33.73|        1.49|         192.0|          -4.3|
|  Staten Island|         65|       916.

In [33]:
spark.sql("""WITH zone_revenue AS (
    SELECT
        pickup_borough,
        pickup_zone,
        SUM(total_amount) AS total_revenue
    FROM taxi_enriched
    GROUP BY pickup_borough, pickup_zone
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY pickup_borough
               ORDER BY total_revenue DESC
           ) AS rn
    FROM zone_revenue
),
top_zones AS (
    SELECT
        pickup_borough,
        pickup_zone
    FROM ranked
    WHERE rn = 1
)
SELECT
    t.pickup_borough,
    t.pickup_zone,
    SUM(e.total_amount) AS total_revenue
FROM taxi_enriched e
JOIN top_zones t
    ON e.pickup_borough = t.pickup_borough
   AND e.pickup_zone = t.pickup_zone
GROUP BY t.pickup_borough, t.pickup_zone
ORDER BY total_revenue DESC;""").show()

+--------------+--------------------+------------------+
|pickup_borough|         pickup_zone|     total_revenue|
+--------------+--------------------+------------------+
|     Manhattan|      Midtown Center|3779643.5799938734|
|       Unknown|                 N/A|  459517.959999875|
|        Queens|         JFK Airport| 292462.4999998996|
|      Brooklyn|Downtown Brooklyn...|61539.430000002285|
|           N/A|      Outside of NYC|10298.799999999972|
|         Bronx|Mott Haven/Port M...| 8851.550000000043|
|           EWR|      Newark Airport|2427.8800000000006|
| Staten Island|Arrochar/Fort Wad...|266.09000000000003|
+--------------+--------------------+------------------+



In [34]:
spark.sql("""SELECT
    CASE
        WHEN HOUR(tpep_pickup_datetime) IN (12, 13, 14) THEN '12–2 PM'
        WHEN HOUR(tpep_pickup_datetime) IN (15, 16) THEN '3–4 PM'
        WHEN HOUR(tpep_pickup_datetime) IN (17, 18, 19) THEN '5–7 PM'
        WHEN HOUR(tpep_pickup_datetime) IN (21, 22) THEN '9–10 PM'
        ELSE 'Other'
    END AS time_block,

    COUNT(*) AS trip_count,

    CASE
        WHEN SUM(total_amount) >= 1000000
            THEN CONCAT(ROUND(SUM(total_amount) / 1000000, 2), 'M')
        WHEN SUM(total_amount) >= 1000
            THEN CONCAT(ROUND(SUM(total_amount) / 1000, 2), 'K')
        ELSE CAST(ROUND(SUM(total_amount), 2) AS STRING)
    END AS revenue_formatted

FROM taxi_enriched
GROUP BY
    CASE
        WHEN HOUR(tpep_pickup_datetime) IN (12, 13, 14) THEN '12–2 PM'
        WHEN HOUR(tpep_pickup_datetime) IN (15, 16) THEN '3–4 PM'
        WHEN HOUR(tpep_pickup_datetime) IN (17, 18, 19) THEN '5–7 PM'
        WHEN HOUR(tpep_pickup_datetime) IN (21, 22) THEN '9–10 PM'
        ELSE 'Other'
    END
ORDER BY SUM(total_amount) DESC;""").show()

+----------+----------+-----------------+
|time_block|trip_count|revenue_formatted|
+----------+----------+-----------------+
|     Other|   2290628|           32.52M|
|    5–7 PM|   1077525|           16.14M|
|   12–2 PM|    905275|           12.57M|
|    3–4 PM|    621745|            8.91M|
|   9–10 PM|    559443|            8.35M|
+----------+----------+-----------------+

